# Level of Effort

:::{div}
:class: colab-badge
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blinkletter/UsingPsi4v2/blob/main/ch2C-level-of-effort.ipynb)
:::


In [2]:
#| label: setup-2C
#| tags: [remove-cell, remove-stdout]

####################################################
### Import Libraries and set up global variables ###
####################################################

from pathlib import Path
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Psi4 cannot be installed with pip. On Colab we first install conda
# (condacolab), which RESTARTS the session. After the restart, run
# this cell again and Psi4 will be installed with conda.
try:
    import psi4
except ImportError:
    if IN_COLAB:
        !pip -q install condacolab
        import condacolab
        condacolab.install()
        !mamba install -q -y -c conda-forge psi4
    import psi4

try:
    import py3Dmol
except ImportError:
    !pip -q install py3Dmol
    import py3Dmol

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Style sheets and helper functions from the book's GitHub repository
if IN_COLAB:
    if not Path("styles/tufte.mplstyle").exists():
        !wget -q -N -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/styles/tufte.mplstyle
        !wget -q -N https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/psi4_helpers.py
    plt.style.use("styles/tufte.mplstyle")
    nproc = 2          # Colab gives you 2 cores
else:
    plt.style.use("styles/tufte.mplstyle")
    nproc = 4          # house limit: 4 threads

import psi4_helpers as ph

# All Psi4 .out and .log files go into the output folder
output_dir = Path("output/ch2/2C-level-of-effort")
output_dir.mkdir(exist_ok=True)

# Resources: house limits are 4 threads and 4 GB
psi4.set_num_threads(nproc)
_ = psi4.set_memory("2 GB")

# 3D pictures: False = static images (as in the book); True = rotatable py3Dmol viewers
INTERACTIVE = True

  Threads set to 4 by Python driver.

  Memory set to   1.863 GiB by Python driver.


Water is so **small** that it needs little effort to produce the calculated results. However, we can **increase** this effort in hopes of achieving greater **accuracy**. Less intense methods make simplifying assumptions or leave out important math that affects the final energy. Are you **treating** the electron pair in an orbital as an electric field (HF and DFT) or are you **keeping track** of the positions of each electron as they try to **avoid** one another within the orbital (MP2, CCSD and others). How much **work** do you want your computer to do?

```{note} The Invisible Code
:class: dropdown
There is a **block** of code **at the top** if each notebook that is **invisible** in this web book. When **running** the notebook in *Colab* or on your own computer, that cell will be visible and needs to be the **first** block **executed**. This setup block at the top of each notebook **sets the resources** that *Psi4* may use. It will sense the environment and, if you are using *Colab*, the code will **install** required *Python* tools. On your own computer you will have set up your *Python* environment in advance according to the [setup instructions](ch1-setup.md).

In this notebook, the hidden code also decalres the **destination** for all output file (`output/ch2/2C-level-of-effort`), sets the **number of processors** to use (2 for *Colab*, 4 for my own computer), and sets **flags** that affect many different cells in the notebook. Here, the flag `INTERACTIVE = False` will make all moelcular images render as **still images**. If you set `INTERACTIVE = True`, then those images will be 3-dimensional **interactive objects** that you can spin around and observe at any angle. The interactive images can be very, **very large** when they include molecular orbitals and other objects. The web book is executed with `INTERACTIVE = False` so that I **don't break** the cloud provider that is serving these pages. You can set it to `True` when running your calculation in *Colab* or on your own machine.
```
No pain, no gain. Every calculation is a choice of **method** (how the electrons interact) and **basis set** (the functions used to build the orbitals). Better choices cost more computer time. In this notebook we optimize water at low, medium and high levels of effort and compare the structures with experiment.

## What You Will Learn
- The main families of **methods**: Hartree–Fock (HF), density functional theory (DFT), MP2 and coupled cluster (CCSD(T)).
- The main families of **basis sets**: minimal (STO-3G), Pople (6-31G*), and correlation-consistent (cc-pVDZ, cc-pVTZ, aug-cc-pVTZ).
- How accuracy and cost change together, and why "more expensive" does not always mean "better" in every property.

## Benchmarking

We need a standard to compare against. We will choose the literature values for bond length and angle. The experimental equilibrium structure of water is $r_{\mathrm{OH}} = 0.9578$ Å and $\angle\mathrm{HOH} = 104.48°$ {cite:p}`HOY19791`. In the [code below](#settings-2C) we will place this information in variables.

In [10]:
#| label: settings-2C
psi4.set_output_file(output_dir / "03-level-of-effort.out", False);

r_exp = 0.9578      # Å
a_exp = 104.48      # degrees

## The Levels We Will Compare

We will be **repeating** the same optimization **calculation** using different "levels of theory". This is not a course in computational chemistry, so I do not plan to explain the details of the choices in [](#tbl-levels). However, you can **read about** these choices in chapter 15 of our **textbook** [@anslyn2006]. 


```{table} A selection of levels of theory for computational chemistry.
:label: tbl-levels
:align: center
| Level | Method / basis | What it adds |
|---|---|---|
| low | HF/STO-3G | Minimal basis: one function per atomic orbital. Fast but crude. |
| low | HF/6-31G | Split valence: two functions for each valence orbital, so orbitals can change size. |
| low | HF/6-31G* | Adds *d* polarization functions on O, so the density can change shape. |
| medium | B3LYP/6-31G* | DFT: includes electron correlation approximately, for about the cost of HF. |
| medium | MP2/cc-pVDZ | Møller–Plesset perturbation theory: the simplest *ab initio* correlation correction. |
| high | MP2/aug-cc-pVTZ | Triple-zeta basis with diffuse ("aug") functions. |
| high | CCSD(T)/cc-pVTZ | Coupled cluster: the "gold standard" for small molecules. |
```
```{note} Psi4 settings used for these levels
:class: dropdown
:label: settings-dropdown
`scf_type: "df"`
: Density fitting for the Hartree–Fock part. **Fast and accurate** to about $10^{-5}$ hartree. It is *Psi4*'s default; we set it explicitly here. *Alternative:* `"pk"` (exact integrals, slower for large basis sets).

`mp2_type: "df"` (default)
: Density-fitted MP2. Much **faster** than conventional MP2 with negligible error. *Alternative:* `"conv"`.

`freeze_core: True`
: Leaves the oxygen 1s electrons out of the **correlation** treatment (MP2, CCSD(T)). The core electrons barely take part in bonding, and the cc-pVXZ basis sets are not designed to describe core correlation, so **freezing** them is both cheaper and **standard** practice. It has no effect on HF or DFT. *Alternative:* `False` together with a core-valence basis set such as cc-pCVTZ, for very high accuracy work.

`cc_type: "conv"` (default for CCSD(T))
: Conventional integrals, required for analytic CCSD(T) gradients in *Psi4*. Density-fitted CCSD(T) (`"df"`) is faster for energies, but its gradients are not available for (T).

DFT integration grid (not set)
: DFT energies are integrated **numerically** on a grid around each atom. *Psi4*'s default grid (75 radial × 302 angular points) is fine for water. *Alternative:* larger grids, `dft_radial_points: 99` and `dft_spherical_points: 590`, for frequencies of floppy molecules.

Other reasonable choices
: `"pbe0/def2-tzvp"` (a modern hybrid functional and a well-balanced Ahlrichs basis set), `"wb97x-d/def2-tzvp"` (range-separated hybrid with dispersion, good for non-covalent interactions), `"b3lyp-d3bj/def2-svp"` (needs the `dftd3` add-on, which is not installed with *Psi4* by default).
```

### Size Matters

The **different** methods use **different** mathematical techniques and assumptions. They take **different** amounts of effort, but the major issue is how they **scale** to the size of the problem. The size is defined by the basis sets. This is a list of atomic **orbitals** included in the calculation and the mathematical **functions** used to approximate the shape of the orbitals. The STO-3G basis set uses only the **valence** atomic orbitals (e.g. s and p for a carbon aton, only s for hydrogen) and three **Gaussian** functions that combine to approximate the **Slater** orbital math that describes each orbital. Computers can **calculate** Gaussian math much, **much faster** that Slater math. The number of functions **increases** with each basis set and the number of atoms {cite:p}`10.1021/acs.jcim.9b00725`. [](#tbl-basis-carbon) and [](#tbl-basis-hydrogen) describe the number of basis functions contained in each set that will be demonstrated in this exercise. **Oxygen** will have the same basis functions as **carbon**; they are on the same row of the periodic table.

````{dropdown} [](#tbl-basis-carbon) and [](#tbl-basis-hydrogen)

```{table} Number of basis functions for a carbon atom in common basis sets.
:label: tbl-basis-carbon
:align: center

| Basis set   | Functions on C | Shells                         | Type                          |
|-------------|:--------------:|--------------------------------|-------------------------------|
| STO-3G      | 5              | 1s, 2s, 2p                     | minimal                       |
| 6-31G       | 9              | 1s, 2s, 2p, 3s, 3p             | split valence                 |
| 6-31G*      | 15             | 6-31G + one *d* set            | split valence + polarization  |
| cc-pVDZ     | 14             | 3*s*, 2*p*, 1*d*               | double zeta + polarization    |
| cc-pVTZ     | 30             | 4*s*, 3*p*, 2*d*, 1*f*         | triple zeta + polarization    |
| aug-cc-pVTZ | 46             | cc-pVTZ + diffuse *s*, *p*, *d*, *f* | adds diffuse functions   |
```
```{table} Number of basis functions for a hydrogen atom in common basis sets.
:label: tbl-basis-hydrogen
:align: center

| Basis set   | Functions on H | Shells                               |
|-------------|:--------------:|--------------------------------------|
| STO-3G      | 1              | 1*s*                                 |
| 6-31G       | 2              | 2*s*                                 |        
| 6-31G*      | 2              | 2*s*                                 |
| cc-pVDZ     | 5              | 2*s*, 1*p*                           |
| cc-pVTZ     | 14             | 3*s*, 2*p*, 1*d*                     |
| aug-cc-pVTZ | 23             | cc-pVTZ + diffuse *s*, *p*, *d*      |
```
````
### Good Stuff isn't Cheap

The methods scale differently with the size of the problem. The lower effort methods scale at *N*{sup}`4` and it can get as bad as *N*{sup}`7` {cite:p}`@https://doi.org/10.1002/anie.202205735`. The level of effort can increase quickly with larger basis sets and larger molecules. [](#tbl-methods) and [](#tbl-methods-pros-cons) provide a brief overview of the methods used in this exercise.

````{dropdown} [](#tbl-methods) and [](#tbl-methods-pros-cons)

```{table} Four common electronic-structure methods, from cheapest to most expensive.
:label: tbl-methods
:align: center

| Method | What it does | Electron correlation | Cost |
|---|---|---|---|
| HF | Hartree–Fock: Each electron moves in the *average* field of all the others; the wavefunction is a single determinant of molecular orbitals. | None | $N^4$ |
| B3LYP | Density Functional Theory: Calculates the energy from the electron density with an approximate exchange–correlation functional; a hybrid that mixes in 20 % HF exchange. | Approximate, through the functional | $N^3$–$N^4$ |
| MP2 | Møller–Plesset, 2nd order: Adds electron correlation to HF by second-order perturbation theory. | Approximate, *ab initio* | $N^5$ |
| CCSD(T) | coupled cluster:CIncludes single and double excitations from HF fully, and triple excitations by perturbation. | Nearly complete, *ab initio* | $N^7$ |
```


```{table} Strengths and weaknesses of the four methods in [](#tbl-methods).
:label: tbl-methods-pros-cons
:align: center

| Method | Strengths | Weaknesses |
|---|---|---|
| HF | Cheap; clear orbital picture; starting point for MP2 and CCSD(T). | Bonds too short, frequencies about 10 % too high; no dispersion; fails for bond breaking (RHF). |
| B3LYP  | About the cost of HF but much more accurate; good structures and frequencies for organic molecules. | No dispersion unless a correction is added (B3LYP-D3); results depend on the functional chosen; cannot be improved systematically. |
| MP2 | Simplest systematic correlation method; includes dispersion; good for hydrogen bonds. | Overestimates dispersion in π-stacked systems; poor when HF is a bad starting point (radicals, stretched bonds). |
| CCSD(T) | The "gold standard": about 4 kJ/mol accuracy for reaction energies with a large basis set. | Very expensive; limited to small molecules; fails for strongly multireference cases such as fully broken bonds. |
```
````

**Ethane** has two carbon atoms and six hydrogens. That means that is has 30 basis set functions when using the **6-31G** basis set and 144 when using the **cc-pVTZ** basis set. If we do the math, an energy calculation using CCSD(T)/cc-pVTZ will take **1.5 billion-fold more** effort than the HF/6-31G. Is it worth the **time**? Is it worth the **money** to buy a bigger computer? This exercise will explore that question using a job small enough to perform on a **laptop computer** in your living room, or a tiny slice of a **data center** in *Google Colab*.

### Are We Clueless?

**Yes we are**. But we are also **fearless**. As you can see, there are a lot of **choices** to make that will change the ability of the calculation to give **accurate** results. The various **settings** and alternative **options** described in the [dropdown bar](#settings-dropdown) above tell me that knowledge of how the **math** works and how the **software** is written is important in using these programs. We will use a handful of **choices** in this exercise, and will use only these **options** throughout this web book. Computational chemistry is like brewing coffee; everybody does it **differently** and everybody knows that everybody else does it **wrong**. I lack the knowledge to get into an **argument** with an expert in this field so I will just admit that **I don't know what I am doing** at the start of every conversation on this subject. **That won't stop me** from using this handful of methods that other have deemed acceptable in simple situations. I am the man without fear; **join me** on the edge.